# RAG Query Pipeline (`query_nbk`)

End-to-end **query-time** RAG pipeline over the `optimized_*` data family.

```
query -> embed (DMRetriever-33M) -> vector search + BM25
      -> RRF fusion -> cross-encoder rerank -> top_k context -> LLM -> answer
```

All paths/table names come from `scripts/settings.py`. Reusable logic lives in `scripts/`;
one-time builds live in `setup_nbk`; incremental ingestion in `ingest_nbk`.


In [0]:
# --- Make scripts/ importable ---
# In Databricks Repos the repo root is on sys.path; locate the scripts/ subdir there
# (works the same when running from the repo root locally).
import os, sys

_cands = [os.path.join(p, "scripts") for p in (list(sys.path) + [os.getcwd()])]
for _cand in _cands:
    if os.path.isdir(_cand) and _cand not in sys.path:
        sys.path.insert(0, _cand)

import warnings
warnings.filterwarnings("ignore")

import markdown
from pyspark.sql import functions as F

import settings
from dbx_hybrid_search import hybrid_search_rrf
from dbx_rerank import rerank_dataframe
from rag_pipeline import rag_pipe_main


def display_MD(md_text):
    # Render Markdown answer nicely in a Databricks notebook
    displayHTML(markdown.markdown(md_text, extensions=["fenced_code", "tables"]))

2026-06-15 21:18:32.545700: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-15 21:18:32.546771: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-06-15 21:18:32.550304: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-06-15 21:18:32.560658: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781558312.576495   11696 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781558312.58

[2026-06-15 21:18:38,260] [WARNING] [real_accelerator.py:194:get_accelerator] Setting accelerator to CPU. If you have GPU or other accelerator, we were unable to detect it.
[2026-06-15 21:18:38,265] [INFO] [real_accelerator.py:239:get_accelerator] Setting ds_accelerator to cpu (auto detect)


/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status


## Config

In [0]:
query = "How did Harris County estimate the number of homes that flooded after Hurricane Harvey, and what data sources did they use?"

TOP_EACH     = 100   # candidates from each retriever before fusion
TOP_N        = 20    # fused candidates kept after RRF (fed into rerank)
RERANK_TOP_K = 10    # final chunks kept after rerank (context for LLM)
RRF_K        = 20    # RRF constant

LLM_MODEL = "gpt-4o-mini"

## Step 1 — Hybrid retrieval (Vector + BM25 -> RRF fusion)

In [0]:
fused_df = hybrid_search_rrf(spark, query, top_each=TOP_EACH, top_n=TOP_N, rrf_k=RRF_K, return_with_text=True)
display(fused_df)

chunk_id,rrf_score,from,text,source_file,chunk_index_in_file
45a2c0d639c28c516cbbafe43b58759c.txt_2,0.047619047619047616,KW,"org/houston‐buyouts/hurricane‐harvey‐home‐buyouts‐harris‐county/ Home buyouts are used by the Flood Control District to reduce flood damages in areas hopelessly deep in the floodplain where structural projects (i. e. channel modifications or detention basins) to reduce flooding are not cost effective and/or beneficial. (HCFCD) Harris Count Flood Control District considers the house’s history of repeated flooding, and only commits to a buyout if the area is at least five acres in size, or if the district can acquire at least 10 connected properties of any size. HCFCD HOME BUYOUT ELIGIBILITY REQUIREMENTS • Source of flooding. • Location and depth within the floodplain. • Cost effectiveness as a solution to the property’s flooding problem. • Potential for future floodplain preservation and/or flood damage reduction projects. • Compatibility with community and natural values. https://www. hcfcd. org/hurricane‐harvey/home‐buyout‐program/ https://www. hcfcd. org/hurricane‐harvey/home‐buyout‐program/ 0.2% Floodplain lowering 1% Floodplain lowering https://www. hcfcd. org/hurricane‐harvey/home‐buyout‐program/ www. uh. edu/class/hobby/harvey/ www. uh. edu/class/hobby/harvey/ Increase in property tax? Increase in sales taxes? www. uh. edu/class/hobby/harvey/",/net/cven-mosta-nas.engr.tamu.edu/volume2/NASdata2/Pie/TDIS_AI_RAG/extract_txt/45a2c0d639c28c516cbbafe43b58759c.txt,2
b5863ae78e3e59d12203c93146df308e.txt_6,0.047619047619047616,VEC,"HOUSE FLOODING ESTIMATES Harvey produced the largest and most devastating house flooding event ever recorded in Harris County. Structure flooding occurred from both overflowing creeks and bayous as well as internal drainage system being overwhelmed by the intense short duration rainfall rates. Based on house flooding assessments, the estimated total number of homes flooded within Harris County is 154,170. Estimated numbers are based on damage and flood assessment reports from most of the cities within Harris County, Harris County Permit Office, Harris County Appraisal District, FEMA flood insurance paid claims, and FEMA Individual Assistance payments for repairs. Duplicates and homes with invalid addresses were removed. Thanks are extended to the cities, Harris County, City of Houston, and FEMA for their hard work locating, assessing damages, and compiling the lists of the tens of thousands of flooded houses. The sheer magnitude of the damage and number of homes, as well as the impact on the residents in this all-time record breaking flood made the job difficult for the flood damage assessment teams. Using the 2016 building footprint data for Harris County indicates that the 154,170 homes flooded during Harvey was 9%-12% of the total number of buildings in the county. Of the 154,170 homes flooded, 48,850 were within the 1% (100-yr) floodplain, 34,970 within the . 2% (500-yr) floodplain, and 70,370 were outside of the 1% (100-yr) and . 2% (500-yr) floodplains. Of the 154,170 homes flooded, 105,340 or 68% were outside the 1% (100-yr) floodplain. The large number of homes flooded outside the 1% (100-yr) floodplain were a result of the extremely high water levels along many creeks and bayous which exceeded the 1% and in some instances the . 2% floodplains as well as intense short duration rainfall rates which resulted in significant flooding from overwhelmed internal drainage systems.",/net/cven-mosta-nas.engr.tamu.edu/volume2/NASdata2/Pie/TDIS_AI_RAG/extract_txt/b5863ae78e3e59d12203c93146df308e.txt,6
4246b23f79a2e7f6e549f9648ab7f3d1.txt_3,0.045454545454545456,KW,"houston and harris county: flooding from hurricane harvey and social vulnerability harvey flooding in texas flooding from harvey flooding and social vulnerability flooding during hurricane harvey intersection of flooding and social vulnerability The flooding devastated many vulnerable towns and neighborhoods. Of the 1,437 c

## Step 2 — Cross-encoder rerank
Requires the reranker model in the UC Volume (see `setup_nbk` step 0).

In [0]:
reranked_df = rerank_dataframe(spark, fused_df, query, top_k=RERANK_TOP_K)
display(reranked_df)

chunk_id,rrf_score,from,text,source_file,chunk_index_in_file,rerank_score
b5863ae78e3e59d12203c93146df308e.txt_6,0.047619047619047616,VEC,"HOUSE FLOODING ESTIMATES Harvey produced the largest and most devastating house flooding event ever recorded in Harris County. Structure flooding occurred from both overflowing creeks and bayous as well as internal drainage system being overwhelmed by the intense short duration rainfall rates. Based on house flooding assessments, the estimated total number of homes flooded within Harris County is 154,170. Estimated numbers are based on damage and flood assessment reports from most of the cities within Harris County, Harris County Permit Office, Harris County Appraisal District, FEMA flood insurance paid claims, and FEMA Individual Assistance payments for repairs. Duplicates and homes with invalid addresses were removed. Thanks are extended to the cities, Harris County, City of Houston, and FEMA for their hard work locating, assessing damages, and compiling the lists of the tens of thousands of flooded houses. The sheer magnitude of the damage and number of homes, as well as the impact on the residents in this all-time record breaking flood made the job difficult for the flood damage assessment teams. Using the 2016 building footprint data for Harris County indicates that the 154,170 homes flooded during Harvey was 9%-12% of the total number of buildings in the county. Of the 154,170 homes flooded, 48,850 were within the 1% (100-yr) floodplain, 34,970 within the . 2% (500-yr) floodplain, and 70,370 were outside of the 1% (100-yr) and . 2% (500-yr) floodplains. Of the 154,170 homes flooded, 105,340 or 68% were outside the 1% (100-yr) floodplain. The large number of homes flooded outside the 1% (100-yr) floodplain were a result of the extremely high water levels along many creeks and bayous which exceeded the 1% and in some instances the . 2% floodplains as well as intense short duration rainfall rates which resulted in significant flooding from overwhelmed internal drainage systems.",/net/cven-mosta-nas.engr.tamu.edu/volume2/NASdata2/Pie/TDIS_AI_RAG/extract_txt/b5863ae78e3e59d12203c93146df308e.txt,6,7.848501682281494
d409350a29070576ce081feb5a635d07.txt_1,0.043478260869565216,VEC,"Over 32,000 residents would be transported to one of 65 temporary shelters in Harris County, where most would wait days until the waters receded to return to damaged homes. It is estimated that over 300,000 vehicles were flooded across Harris County. The Harris County Medical Examiner’s Office confirms 36 flood related deaths in the county, including several people drowning in their home or work place. Multiple sources were used to analyze flood damage impact for this report. Data sources included: FEMA Individual Assistance (IA) data, Harris County Engineering Department (HCED), Harris County Flood Control District (HCFCD), U. S. Army Corps of Engineers (USACE), Harris County Appraisal District (HCAD), Harris County Public Health (HCPH) and other agencies servicing Harris County. Data was also collected through various citizen engagement activities that included community meetings, surveys, small focus group meetings, social media, canvasing, and direct mail. The goals for community engagement were to engage the public, especially vulnerable populations such as low-income and persons with a disability; housing and civil rights advocates; local community leaders; non-profit organizations; business owners; and other area stakeholders. The Needs Assessment is dependent on information gathered for Unincorporated Harris County and the thirty-three (33) cities within Harris County that agreed to participate in this assessment. The sections below provide more detail on the various types of damage assessed and collected. 2. Housing Hurricane Harvey produced the most devastating house flooding ever recorded in Harris County. Based on a house flooding assessment report by the Harris County Flood Control District (HCFCD), abo

## Step 3 — LLM generation
`rag_pipe_main` runs retrieve -> fuse -> rerank -> generate in one call.

In [0]:
reranked_df, answer = rag_pipe_main(
    spark, query,
    llm_model=LLM_MODEL, use_dbx_model=False, temperature=0.2,
    top_each=TOP_EACH, top_n=TOP_N, rerank_top_k=RERANK_TOP_K, rrf_k=RRF_K,
)
display_MD(answer)

LLM prompting done.


Answers: Harris County estimated the number of homes that flooded after Hurricane Harvey based on damage and flood assessment reports from various cities within the county, the Harris County Permit Office, Harris County Appraisal District, FEMA flood insurance paid claims, and FEMA Individual Assistance payments for repairs. Duplicates and homes with invalid addresses were removed to arrive at the final estimate. 
 Data sources: Damage and flood assessment reports, Harris County Permit Office, Harris County Appraisal District, FEMA flood insurance claims, FEMA Individual Assistance payments, community engagement activities (meetings, surveys, focus groups, social media, canvasing, direct mail).

Trace(trace_id=tr-9b81e0337e8c0505ce68dab72a7e3d46)

## Batch evaluation
Runs the pipeline over a fixed QA set across several models and writes answers to `settings.QA_EVAL_TABLE`.

In [0]:
qa_dict = [
    {"question": "How did the 2021 winter storm impact Austin's infrastructure in terms of power and electricity",
     "fact": "Approximately 40% of Austin Energy customers were without power.",
     "fact_source": "2021 Winter Storm Uri After-Action Review"},
    {"question": "What were the major barriers to effective communication for people with disabilities during Hurricane Harvey",
     "fact": "Barriers included lack of Text-to-911, system overloads, dependence on power, captioning failures, limited interpreter availability, and restricted access to information.",
     "fact_source": "Hurricane Harvey After Action Report on Individuals with Disabilities"},
    {"question": "What are the four Readiness Levels defined in the Brazos County Emergency Management Plan",
     "fact": "Level 4: Normal Conditions; Level 3: Increased Readiness; Level 2: High Readiness; Level 1: Maximum Readiness (e.g., tropical weather threat, tornado warning, flash flood).",
     "fact_source": "Brazos County Emergency Management Plan"},
    {"question": "What issues were identified regarding contractor safety during the storm Mara?",
     "fact": "Contractors exhibited unsafe behaviors. Recommended actions include pre-approval of contractor safety plans and formal processes to document, report, and correct unsafe behavior. A related finding noted a defective utility pole previously tagged as unsafe contributed to the Smokehouse Creek Fire.",
     "fact_source": "Winter Storm Mara: After Action Report"},
    {"question": "What is CRISP program in Austin?",
     "fact": "CRISP stands for the Community Resiliency Improvement Status Portal, developed by the Austin and Travis County Emergency Management Offices to provide transparency into disaster response and recovery efforts.",
     "fact_source": "Austin CRISP"},
    {"question": "How does the CRISP dashboard track the status of After-Action Report recommendations",
     "fact": "Statuses include Completed (verified finished), In Progress (with percentage completion), Not Started, On Hold or Pending (awaiting assignment or resources), and Closed (cannot be completed or merged into another item).",
     "fact_source": "Austin CRISP"},
    {"question": "What location was used for a full-scale county-wide POD exercise in July 2017 to simulate recovery from a Category 4 hurricane",
     "fact": "The NRG Center was used for the full-scale POD exercise.",
     "fact_source": "Harris County Tests Points of Distribution Plan"},
    {"question": "How did Hurricane Harvey flooding impact POD locations in Houston?",
     "fact": "Flooding made many POD sites and routes inaccessible, reduced effectiveness of pre-selected locations, and required post-incident site authorization. Studies indicate flood-aware POD planning could reduce travel distance by 46.5%. Infrastructure failures further constrained viable POD locations.",
     "fact_source": "https://pubmed.ncbi.nlm.nih.gov/32441037/"},
    {"question": "What specific equipment is required for a Type I POD?",
     "fact": "Required equipment includes 3 forklifts, 3 pallet jacks, 2 power light sets, 6 portable toilets, 2 tents, 4 dumpsters, 30 traffic cones, and 4 two-way radios.",
     "fact_source": "Harris County Points of Distribution (POD) Plan"},
    {"question": "What are the three core components used to assess flood risk in the 2024 State Flood Plan",
     "fact": "Flood risk is assessed using Flood Hazard (magnitude, location, frequency), Potential Exposure (people and property in hazard areas), and Vulnerability (degree of impact on communities and facilities).",
     "fact_source": "2024 State Flood Plan | Texas Water Development Board"},
    {"question": "How many Texans currently live in a 100-year floodplain?",
     "fact": "Approximately 2.4 million Texans live or work within the 1 percent (100-year) annual chance floodplain.",
     "fact_source": "2024 State Flood Plan | Texas Water Development Board"},
    {"question": "What is the Galveston Bay Surge Protection project's total cost?",
     "fact": "The Coastal Texas Project is authorized at $34.38B. The Galveston Bay Storm Surge Barrier System accounts for $31.20B, with a 65% federal share and 35% non-federal share.",
     "fact_source": "2024 State Flood Plan | Texas Water Development Board"},
    {"question": "How do playa improvements help reduce state flood risk?",
     "fact": "Playa improvements mitigate flat-terrain ponding through region-specific, nature-based and structural flood mitigation solutions.",
     "fact_source": "Technical Guidelines for Regional Flood Planning | TWDB"},
    {"question": "Tell me more about the strategic buyout program in Onion Creek.",
     "fact": "The voluntary buyout program addresses repeated flooding by purchasing homes along Upper Onion Creek, removing structures, restoring land through regrading and revegetation, and preserving it as permanent open space with potential community benefits.",
     "fact_source": "Nature-based solution Hill Country documentation"},
    {"question": "How does the flat terrain of the Houston area influence the choice of hydraulic modeling software used for flood planning",
     "fact": "Flat coastal terrain causes widespread, multi-directional flooding requiring 2D hydraulic analysis. Houston-area planning commonly uses HEC-RAS 2D or XPSWMM, with 1D/2D coupling or alternatives needed to represent underground drainage systems.",
     "fact_source": "Technical Guidelines for Regional Flood Planning | TWDB"},
]

In [0]:
from tqdm import tqdm
import mlflow
mlflow.autolog(disable=True)

EVAL_MODEL = "gpt-4o-mini"
for item in tqdm(qa_dict, desc=f"QA | {EVAL_MODEL}"):
    q = item["question"]
    _, ans = rag_pipe_main(spark, q, llm_model=EVAL_MODEL, use_dbx_model=False, temperature=0.2)
    item[f"answer_{EVAL_MODEL}"] = ans

QA | gpt-4o-mini:   7%|▋         | 1/15 [00:19<04:39, 19.96s/it]

LLM prompting done.


QA | gpt-4o-mini:  13%|█▎        | 2/15 [00:34<03:39, 16.86s/it]

LLM prompting done.


QA | gpt-4o-mini:  20%|██        | 3/15 [00:48<03:05, 15.43s/it]

...

QA | gpt-4o-mini:  93%|█████████▎| 14/15 [03:15<00:14, 14.83s/it]

LLM prompting done.


QA | gpt-4o-mini: 100%|██████████| 15/15 [03:28<00:00, 13.91s/it]

LLM prompting done.


[Trace(trace_id=tr-198582794ab715de8c65b3442d4d85b3), Trace(trace_id=tr-471f6905c1185d8f2598c0619b179ce8), Trace(trace_id=tr-577e511300de4a7d30438a8dd829bc44), Trace(trace_id=tr-500135d76260760e80c7cbb53232bb26), Trace(trace_id=tr-3b019362e7a86325c02b34f2e0136aab), Trace(trace_id=tr-3f2df1a366b3972a1a15b2afa5ba4298), Trace(trace_id=tr-3bdfd7944d72c0354ab1e093f9d5ed32), Trace(trace_id=tr-32eaf735a5545bf74e2918eced867601), Trace(trace_id=tr-8590f2bea386c08ab1321d58a79d6557), Trace(trace_id=tr-dfa64aa96a80bf4e3a3d24543ca27479)]

In [0]:
import json
from pyspark.sql import Row

def _to_str(v):
    if v is None: return None
    if isinstance(v, (str, int, float, bool)): return str(v)
    return json.dumps(v, ensure_ascii=False)

qa_rows = [{k.replace("-", "_"): _to_str(v) for k, v in r.items()} for r in qa_dict]
eval_df = spark.createDataFrame([Row(**x) for x in qa_rows])
eval_df.write.mode("overwrite").format("delta").option("mergeSchema", "true").saveAsTable(settings.QA_EVAL_TABLE)
display(eval_df)

question,fact,fact_source,answer_gpt_4o_mini
How did the 2021 winter storm impact Austin's infrastructure in terms of power and electricity,Approximately 40% of Austin Energy customers were without power.,2021 Winter Storm Uri After-Action Review,"- Answers: The 2021 winter storm caused significant strain on Austin's power infrastructure, leading to widespread power outages and disruptions. Austin Energy's generation assets largely remained operational, but the extreme cold resulted in frozen equipment and infrastructure failures. Many residents experienced prolonged outages, and the city had to implement a boil-water notice due to disruptions in water distribution systems. The storm highlighted vulnerabilities in the infrastructure, prompting Austin Energy to investigate measures to improve resilience, such as burying power lines and enhancing operational resilience. - Data sources: [""Beginning on February 14, 2021, a series of winter storms brought record-breaking low temperatures and snowfall to all of Texas..."", ""Winter Storm Uri demonstrated perfectly just how challenging extreme cold can be..."", ""Additionally, several reports following the winter storm have indicated the failure of Texas’s power grid could have been far more catastrophic if Austin Energy and other ERCOT participants did not comply with the outage mandates..."", ""Austin Energy will investigate measures to improve its infrastructure such as conducting a third-party study of burying power lines...""]"
What were the major barriers to effective communication for people with disabilities during Hurricane Harvey,"Barriers included lack of Text-to-911, system overloads, dependence on power, captioning failures, limited interpreter availability, and restricted access to information.",Hurricane Harvey After Action Report on Individuals with Disabilities,"- Answers: Major barriers to effective communication for people with disabilities during Hurricane Harvey included: lack of accessible communication methods (e.g., closed captioning and audio descriptions), insufficient training for emergency services personnel on how to interact with people with disabilities, reliance on television and radio for critical information which was often not accessible, and communication breakdowns that left individuals unaware of evacuation instructions and emergency updates. - Data sources: [1], [2], [3], [4], [5], [6]"
What are the four Readiness Levels defined in the Brazos County Emergency Management Plan,"Level 4: Normal Conditions; Level 3: Increased Readiness; Level 2: High Readiness; Level 1: Maximum Readiness (e.g., tropical weather threat, tornado warning, flash flood).",Brazos County Emergency Management Plan,- Answers: 1. Level 4: Normal Conditions 2. Level 3: Increased Readiness 3. Level 2: Not specified in the provided context. 4. Level 1: Not specified in the provided context. - Data sources: The context provided does not specify all four readiness levels for Brazos County.
What issues were identified regarding contractor safety during the storm Mara?,"Contractors exhibited unsafe behaviors. Recommended actions include pre-approval of contractor safety plans and formal processes to document, report, and correct unsafe behavior. A related finding noted a defective utility pole previously tagged as unsafe contributed to the Smokehouse Creek Fire.",Winter Storm Mara: After Action Report,"- Answers: The restoration efforts during Winter Storm Mara raised serious safety concerns for utility workers navigating hazardous conditions, including power lines coated in ice and transformers struggling under the weight of ice. Open communication between contractor and utility operations was critical for success, and coordination between contractor supervision and utility operations was necessary to manage the ""unknowns"" effectively. - Data sources: Winter Storm Mara Case Study and Lessons Learned, Winter Storm Mara highlighted areas of improvement relating to tree trimming/ve